# ZBUE: Milestone 2 — Income Estimation Model
> **Zeyro Behavioral Underwriting Engine**  
> Goal: Predict `monthly_income` from behavioral transaction features.  
> Models: Linear Regression · Random Forest · LightGBM · XGBoost  
> Metrics: MAE · RMSE · MAPE · R²  
> Tracking: MLflow (`ZBUE_Income_Estimation` experiment)

In [1]:
import sys, os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
import pickle
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.float_format', '{:.4f}'.format)
print("Environment ready.")

Environment ready.


## 1. Load Feature Store

In [2]:
df = pd.read_parquet('../retail_lending/data/processed/feature_store.parquet')
print(f"Feature Store: {df.shape[0]} clients x {df.shape[1]} features")
display(df.describe().T)

Feature Store: 4941 clients x 30 features


,count,mean,std,min,25%,50%,75%,max
monthly_income,4941.0000,53516.1569,32804.5874,12500.0000,30410.4167,45031.8333,66377.2500,250000.0000
yearly_income,4941.0000,642193.8828,393655.0487,150000.0000,364925.0000,540382.0000,796527.0000,3000000.0000
income_per_capita,4941.0000,53516.1548,32804.5927,12500.0000,30410.0000,45032.0000,66377.0000,250000.0000
credit_score,4941.0000,701.9247,78.6333,421.0000,647.0000,700.0000,757.0000,900.0000
total_debt,4941.0000,119974.8628,162620.4018,0.0000,0.0000,68779.0000,172721.0000,1817384.0000
num_credit_cards,4941.0000,2.3576,1.1695,1.0000,1.0000,2.0000,3.0000,5.0000
age,4941.0000,44.2309,14.9141,19.0000,31.0000,44.0000,57.0000,70.0000
tx_frequency_per_month,4941.0000,0.4423,0.3896,0.0592,0.1788,0.2742,0.5687,3.0000
total_spend,4941.0000,12476.3688,10406.9455,157.9500,4726.9500,9613.5900,17033.1800,67036.1300
avg_tx_amount,4941.0000,3074.1666,1383.0858,157.9500,2171.9700,2944.5514,3718.1000,18754.2100


## 2. Prepare Features & Target

In [3]:
TARGET = 'monthly_income'

# Drop leaky and non-feature columns
DROP_COLS = ['client_id', 'monthly_income', 'yearly_income', 'income_per_capita', 'disposable_income', 'savings_rate']
feature_cols = [c for c in df.columns if c not in DROP_COLS]

X = df[feature_cols].fillna(df[feature_cols].median())
y = df[TARGET]

print(f"Features ({len(feature_cols)}): {feature_cols}")
print(f"Target  : ₹{y.min():,.0f} → ₹{y.max():,.0f}  (mean ₹{y.mean():,.0f})")

Features (24): ['credit_score', 'total_debt', 'num_credit_cards', 'age', 'tx_frequency_per_month', 'total_spend', 'avg_tx_amount', 'max_tx_amount', 'min_tx_amount', 'tx_count', 'chip_usage_ratio', 'merchant_diversity', 'city_diversity', 'spending_entropy', 'spending_volatility', 'weekend_spend_ratio', 'mom_spend_volatility', 'recency_days', 'customer_tenure_days', 'active_months', 'monthly_spend_est', 'foir', 'dti', 'debt_per_card']
Target  : ₹12,500 → ₹250,000  (mean ₹53,516)


## 3. Train/Test Split

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape[0]:,}  |  Test: {X_test.shape[0]:,}")

Train: 3,952  |  Test: 989


## 4. Metrics Helper

In [5]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

results = []

def evaluate(name, y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    r2   = r2_score(y_true, y_pred)
    print(f"  {name:<25}  MAE: ₹{mae:>10,.0f}  RMSE: ₹{rmse:>10,.0f}  MAPE: {mape:>6.2f}%  R²: {r2:.4f}")
    results.append({"Model": name, "MAE": mae, "RMSE": rmse, "MAPE": mape, "R2": r2})
    return mae, rmse, mape, r2

## 5. Model Training

In [6]:
import mlflow
mlflow.set_tracking_uri('file:../mlruns')
mlflow.set_experiment('ZBUE_Income_Estimation')
print("MLflow ready.")

MLflow ready.


In [7]:
# Model 1: Linear Regression
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

with mlflow.start_run(run_name="LinearRegression"):
    pipe = Pipeline([('scaler', StandardScaler()), ('model', LinearRegression())])
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    mae, rmse, mape, r2 = evaluate("Linear Regression", y_test, preds)
    mlflow.log_metrics({"mae": mae, "rmse": rmse, "mape": mape, "r2": r2})
    print("  ✅ Logged to MLflow")

  Linear Regression          MAE: ₹    11,957  RMSE: ₹    18,667  MAPE:  28.19%  R²: 0.6570
  ✅ Logged to MLflow


In [8]:
# Model 2: Random Forest
from sklearn.ensemble import RandomForestRegressor

with mlflow.start_run(run_name="RandomForest"):
    rf = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    preds = rf.predict(X_test)
    mae, rmse, mape, r2 = evaluate("Random Forest", y_test, preds)
    mlflow.log_params({"n_estimators": 100, "max_depth": 10})
    mlflow.log_metrics({"mae": mae, "rmse": rmse, "mape": mape, "r2": r2})
    print("  ✅ Logged to MLflow")

  Random Forest              MAE: ₹     3,262  RMSE: ₹     5,639  MAPE:   8.24%  R²: 0.9687
  ✅ Logged to MLflow


In [9]:
# Model 3: LightGBM
import lightgbm as lgb

with mlflow.start_run(run_name="LightGBM"):
    lgbm = lgb.LGBMRegressor(n_estimators=200, learning_rate=0.05, max_depth=6,
                              num_leaves=31, random_state=42, verbose=-1)
    lgbm.fit(X_train, y_train, eval_set=[(X_test, y_test)], callbacks=[lgb.log_evaluation(-1)])
    preds = lgbm.predict(X_test)
    mae, rmse, mape, r2 = evaluate("LightGBM", y_test, preds)
    mlflow.log_params({"n_estimators": 200, "learning_rate": 0.05, "max_depth": 6})
    mlflow.log_metrics({"mae": mae, "rmse": rmse, "mape": mape, "r2": r2})
    print("  ✅ Logged to MLflow")

  LightGBM                   MAE: ₹     2,594  RMSE: ₹     4,629  MAPE:   4.99%  R²: 0.9789
  ✅ Logged to MLflow


In [10]:
# Model 4: XGBoost
from xgboost import XGBRegressor

with mlflow.start_run(run_name="XGBoost"):
    xgb = XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=6,
                       subsample=0.8, colsample_bytree=0.8, random_state=42, verbosity=0)
    xgb.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
    preds = xgb.predict(X_test)
    mae, rmse, mape, r2 = evaluate("XGBoost", y_test, preds)
    mlflow.log_params({"n_estimators": 200, "learning_rate": 0.05, "max_depth": 6})
    mlflow.log_metrics({"mae": mae, "rmse": rmse, "mape": mape, "r2": r2})
    print("  ✅ Logged to MLflow")

  XGBoost                    MAE: ₹     2,310  RMSE: ₹     3,793  MAPE:   4.93%  R²: 0.9858
  ✅ Logged to MLflow


## 6. Benchmark Comparison Table

In [11]:
results_df = pd.DataFrame(results).set_index('Model').sort_values('MAPE')

display_df = results_df.copy()
display_df['MAE']  = display_df['MAE'].apply(lambda x: f"₹{x:,.0f}")
display_df['RMSE'] = display_df['RMSE'].apply(lambda x: f"₹{x:,.0f}")
display_df['MAPE'] = display_df['MAPE'].apply(lambda x: f"{x:.2f}%")
display_df['R2']   = display_df['R2'].apply(lambda x: f"{x:.4f}")
display_df.columns = ['MAE ↓', 'RMSE ↓', 'MAPE ↓', 'R² ↑']

print("\n=== ZBUE Income Estimation — Model Benchmark ===")
display(display_df)

champion_name = results_df['MAPE'].idxmin()
champion = results_df.loc[champion_name]
print(f"\n🏆 Champion: {champion_name}  MAPE: {champion['MAPE']:.2f}%  R²: {champion['R2']:.4f}")


=== ZBUE Income Estimation — Model Benchmark ===


,MAE ↓,RMSE ↓,MAPE ↓,R² ↑
Model,,,,
XGBoost,"₹2,310","₹3,793",4.93%,0.9858
LightGBM,"₹2,594","₹4,629",4.99%,0.9789
Random Forest,"₹3,262","₹5,639",8.24%,0.9687
Linear Regression,"₹11,957","₹18,667",28.19%,0.6570



🏆 Champion: XGBoost  MAPE: 4.93%  R²: 0.9858


## 7. Feature Importance

In [12]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Use champion model for feature importance
models_map = {"XGBoost": xgb, "LightGBM": lgbm, "Random Forest": rf}

if champion_name in models_map:
    champ_model = models_map[champion_name]
    fi = pd.Series(champ_model.feature_importances_, index=feature_cols).sort_values()
    
    fig, ax = plt.subplots(figsize=(8, 6))
    fi.plot(kind='barh', ax=ax, color='#4A90D9')
    ax.set_title(f'Feature Importance — {champion_name}', fontweight='bold', fontsize=13)
    ax.set_xlabel('Importance Score')
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.savefig('../retail_lending/data/processed/feature_importance.png', dpi=120, bbox_inches='tight')
    plt.show()
    print("Feature importance saved.")
else:
    print("Linear Regression selected — no feature importances available.")

Feature importance saved.


## 8. Save Champion Model

In [13]:
models_map_full = {"XGBoost": xgb, "LightGBM": lgbm, "Random Forest": rf, "Linear Regression": pipe}
champion_model = models_map_full[champion_name]

model_path = '../retail_lending/data/processed/income_model.pkl'
with open(model_path, 'wb') as f:
    pickle.dump({
        'model': champion_model,
        'feature_cols': feature_cols,
        'champion_name': champion_name,
        'mape': results_df.loc[champion_name, 'MAPE'],
        'r2':   results_df.loc[champion_name, 'R2']
    }, f)

print(f"✅ Saved: {model_path}")
print(f"   Model : {champion_name}")
print(f"   MAPE  : {results_df.loc[champion_name, 'MAPE']:.2f}%")
print(f"   R²    : {results_df.loc[champion_name, 'R2']:.4f}")

✅ Saved: ../retail_lending/data/processed/income_model.pkl
   Model : XGBoost
   MAPE  : 4.93%
   R²    : 0.9858


## Summary
> **Milestone 2 Complete.** Champion model saved to `income_model.pkl`.  
> Proceed to **Milestone 3**: Lead Conversion Baseline on UCI Bank Marketing.